<a href="https://colab.research.google.com/github/AnkaRezaa/machine_learning_254107023002/blob/main/JS15/Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas: Deployment Model Proyek ke Hugging Face (Docker + Flask)

### Identitas Mahasiswa:
- **Nama**  : MUHAMMAD ANKA REZA
- **NIM**   : 254107023002
- **Kelas** : TI/3H

---

### Deskripsi Tugas:
Setelah berhasil melakukan deployment pada studi kasus latihan ("Day vs Night"), tugas selanjutnya adalah menerapkan pipeline deployment yang sama untuk model **Project Anda masing-masing**.

**Ketentuan Berkas Proyek:**
1. **File Model (`.h5`):** Model yang sudah dilatih dengan akurasi terbaik.
2. **File Scaler (`.pkl`):** Objek scaler atau preprocessor yang digunakan saat training.
3. **File `app.py`:** Aplikasi web server Flask yang sesuai dengan logika model Anda.
4. **File `requirements.txt`:** Daftar library yang digunakan.
5. **File `Dockerfile`:** Konfigurasi container Docker untuk lingkungan Hugging Face Spaces.

### Langkah 1: Menyiapkan Folder Proyek dan Artefak Model
Membuat direktori kerja baru bernama `myproject-deployment` dan memastikan file artefak model (`.h5`) serta scaler/preprocessor (`.pkl`) telah tersedia di dalamnya.

In [2]:
import os
import shutil

# Tentukan nama folder deployment proyek Anda
project_folder = 'myproject-deployment'
os.makedirs(project_folder, exist_ok=True)

print(f"✅ Folder '{project_folder}' siap digunakan.")

✅ Folder 'myproject-deployment' siap digunakan.


### Langkah 2: Membuat File Server `app.py`
Menuliskan script server Flask `app.py` yang memuat model `.h5`, melakukan pra-pengolahan data/citra masukan, dan mengembalikan hasil prediksi melalui antarmuka web pada port `7860`.

In [3]:
app_py_code = '''import os
import numpy as np
import cv2
import pickle
import tensorflow as tf
from flask import Flask, request, render_template_string

app = Flask(__name__)

# Path Model & Scaler Proyek Anda
MODEL_PATH = 'model.h5'
SCALER_PATH = 'scaler.pkl'

model = None
scaler = None

try:
    if os.path.exists(MODEL_PATH):
        model = tf.keras.models.load_model(MODEL_PATH)
        print("✅ Model loaded successfully.")
    if os.path.exists(SCALER_PATH):
        with open(SCALER_PATH, 'rb') as f:
            scaler = pickle.load(f)
        print("✅ Scaler loaded successfully.")
except Exception as e:
    print(f"❌ Error loading model/scaler: {e}")

@app.route('/', methods=['GET'])
def home():
    return render_template_string("""
    <!DOCTYPE html>
    <html>
    <head>
        <title>Machine Learning Project Classifier</title>
        <style>
            body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #f4f6f9; text-align: center; padding: 50px; }
            .container { background: white; max-width: 500px; margin: auto; padding: 30px; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.1); }
            h2 { color: #333; margin-bottom: 20px; }
            input[type=file] { margin: 20px 0; }
            button { background-color: #007bff; color: white; border: none; padding: 10px 24px; font-size: 16px; border-radius: 6px; cursor: pointer; }
            button:hover { background-color: #0056b3; }
        </style>
    </head>
    <body>
        <div class="container">
            <h2>ML Project Classifier</h2>
            <p>Silakan unggah berkas citra untuk melakukan inferensi prediksi.</p>
            <form action="/predict" method="post" enctype="multipart/form-data">
                <input type="file" name="file" accept="image/*" required><br>
                <button type="submit">Prediksi Citra</button>
            </form>
        </div>
    </body>
    </html>
    """)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        file = request.files['file']
        nparr = np.frombuffer(file.read(), np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        
        # Sesuaikan dengan target ukuran citra model Anda
        img_resized = cv2.resize(img, (64, 64)) / 255.0
        img_input = np.expand_dims(img_resized, axis=0)
        
        prediction = model.predict(img_input)
        prob = float(prediction[0][0]) if prediction.shape[-1] == 1 else float(np.max(prediction[0]))
        result_label = f"Hasil Prediksi (Probabilitas: {prob:.4f})"
        
        return f"<div style='text-align:center; padding:50px;'><h2>{result_label}</h2><br><a href='/'>Kembali</a></div>"
    except Exception as e:
        return f"<div style='text-align:center; padding:50px;'><h3>Terjadi Kesalahan: {e}</h3><br><a href='/'>Kembali</a></div>"

if __name__ == '__main__':
    # Port 7860 wajib untuk Hugging Face Spaces
    app.run(host='0.0.0.0', port=7860)
'''

with open(os.path.join(project_folder, 'app.py'), 'w', encoding='utf-8') as f:
    f.write(app_py_code)

print("✅ File app.py berhasil dibuat di folder deployment.")

✅ File app.py berhasil dibuat di folder deployment.


### Langkah 3: Membuat File `requirements.txt`
Mendaftarkan seluruh pustaka (*library*) yang dibutuhkan server container saat dijalankan di cloud.

In [4]:
requirements_code = '''flask
tensorflow-cpu
numpy
scikit-learn
opencv-python-headless
gunicorn
'''

with open(os.path.join(project_folder, 'requirements.txt'), 'w', encoding='utf-8') as f:
    f.write(requirements_code)

print("✅ File requirements.txt berhasil dibuat.")

✅ File requirements.txt berhasil dibuat.


### Langkah 4: Membuat File `Dockerfile`
Menyusun instruksi pembuatan container Docker berbasis `python:3.9-slim` dengan konfigurasi non-root user dan port `7860` standar Hugging Face Spaces.

In [5]:
dockerfile_code = '''# Base Image Python 3.9 Slim
FROM python:3.9-slim

# Install Library Sistem untuk OpenCV (Debian Compatible)
RUN apt-get update && apt-get install -y \
    libgl1 \
    libglib2.0-0 \
    && rm -rf /var/lib/apt/lists/*

# Setup User Non-Root (Standar Keamanan Hugging Face Spaces)
RUN useradd -m -u 1000 user
USER user
ENV PATH="/home/user/.local/bin:$PATH"

# Setup Direktori Kerja
WORKDIR /app

# Install Dependencies
COPY --chown=user ./requirements.txt requirements.txt
RUN pip install --no-cache-dir --upgrade -r requirements.txt

# Copy Seluruh Berkas Aplikasi ke Direktori Kerja
COPY --chown=user . /app

# Expose Port & Jalankan Aplikasi
EXPOSE 7860
CMD ["python", "app.py"]
'''

with open(os.path.join(project_folder, 'Dockerfile'), 'w', encoding='utf-8') as f:
    f.write(dockerfile_code)

print("✅ File Dockerfile berhasil dibuat.")

✅ File Dockerfile berhasil dibuat.


### Langkah 5: Setup Space Baru di Hugging Face
1. Buka dan login ke akun [Hugging Face](https://huggingface.co/).
2. Buat Space baru melalui menu **New Space**:
   - **Space Name:** `mlproject-classifier-254107023002` (atau sesuaikan dengan nama project Anda).
   - **License:** `Apache 2.0` / `MIT`.
   - **Space SDK:** Pilih **Docker**.
   - **Template:** Kosongi (*blank*).
   - **Visibility:** `Public`.
   - Klik tombol **Create Space**.
3. Buat **Access Token** izin tulis (*Write*):
   - Buka **Settings > Access Tokens** pada profil Hugging Face Anda.
   - Klik **Create new token**, pilih Role **Write**, lalu salin token tersebut.

### Langkah 6: Proses Git Deployment ke Hugging Face

Jalankan langkah-langkah berikut di terminal (Git Bash / PowerShell):

```bash
# 1. Clone repository Space kosong dari Hugging Face
git clone https://huggingface.co/spaces/USERNAME_ANDA/NAMA_SPACE_ANDA

# 2. Salin seluruh file dari folder 'myproject-deployment' ke dalam folder hasil clone tersebut:
#    - app.py
#    - Dockerfile
#    - requirements.txt
#    - model.h5
#    - scaler.pkl

# 3. Masuk ke folder repository hasil clone
cd NAMA_SPACE_ANDA

# 4. Lakukan git add, commit, dan push
git add .
git commit -m "Deploy Project ML ke Hugging Face"
git push
```

> **Perhatian saat autentikasi:**
> Masukkan **Username** akun Hugging Face Anda, dan pada kolom **Password**, tempelkan (*paste*) **Access Token (Write)** yang telah dibuat sebelumnya.

### Langkah 7: Pengujian, Laporan Deployment, dan Link Aplikasi

Setelah status Space di Hugging Face berubah menjadi **Running** (Warna Hijau):

- **Nama Proyek:** *(Tuliskan nama proyek ML Anda, misal: Klasifikasi Citra Cat vs Dog / CIFAR-10 / Day vs Night)*
- **URL Hugging Face Space:** `https://huggingface.co/spaces/USERNAME_ANDA/NAMA_SPACE`
- **Status Deployment:** Running (Docker Container)
- **Dokumentasi Hasil Pengujian:**
  *(Lampirkan tangkapan layar/screenshot antarmuka aplikasi web Hugging Face saat berhasil memprediksi data/citra).* 